# Task 2 — Treeplexes, Sequence Form, and Best Responses (20%)

## Objectives

By the end of this task you should be able to:

- Understand the treeplex representation of the sequence-form strategy space.
- Convert between behavioral and sequence-form strategies.
- Compute the best response of one player to a fixed opponent sequence-form strategy.
- Compute the saddle-point residual (exploitability) of a strategy profile.

## Relevant Files

| File | Role |
|------|------|
| `student_attempts/treeplex_representation.py` | `Treeplex`, `TreeplexGame`, behavioral ↔ sequence-form conversion |
| `student_attempts/solver.py` | `Solver` base class — **edit this file** to implement the two tasks |
| `student_attempts/env_config.py` | `env`, `PLAYER_1`, `PLAYER_2`, `make_round_start_state` |
| `tasks/task2.ipynb` | This notebook — run cells top to bottom |

> **Working directory note:** All `student_attempts/` modules import from each other using bare names (e.g. `from env_config import ...`). The setup cell below adds `student_attempts/` to the Python path so those imports resolve correctly.

## Section 1 — The Treeplex Representation

### Sequences

A **sequence** for player $i$ is the sequence of infoset-actions player $i$ has taken along a root-to-node path. Because of perfect recall, two nodes belong to the same information set if and only if player $i$ has played exactly the same sequence of actions to reach them.

We index sequences with integers. Index **0** is the *empty sequence* — player $i$ has not yet acted. Recall from class that the empty sequence is this "dummy" sequence symbolizing the start of the game where neither player has taken any action.

### The `Treeplex` class

A `Treeplex` encodes all sequences for one player and how they relate to each other:

| Field | Meaning |
|-------|---------|
| `num_sequences` | Total number of sequences (1 + sum of actions across all info sets) |
| `info_state_start_seq[i]` | First sequence index for info set $i$ |
| `info_state_end_seq[i]` | One past the last sequence index for info set $i$ |
| `info_state_parent_seq[i]` | The sequence the player played to *reach* info set $i$ |

We have enforced during construction of the treeplex that sequences belonging to each information set have contiguous indices. Therefore, infoset $i$ owns the sequences in the range info_state_start[i], ... info_state_end[i]-1. Note that we have not included info_state_end[i] itself. So the actions available at info set $i$ correspond to sequence indices `start_seq[i] : end_seq[i]` and can be looped over conveniently.

### The `TreeplexGame` class

`TreeplexGame` builds both players' treeplexes in a single DFS traversal of the live game state, and simultaneously accumulates the payoff structure:

- `seq_pair_to_leaves[(s1, s2)]` — the chance-weighted terminal payoff for Player 1 when the game reaches the leaf via sequence pair $(s_1, s_2)$. This encodes the full **bilinear payoff**:
$$u_1(x_1, x_2) = \sum_{(s_1, s_2)} x_1[s_1] \cdot x_2[s_2] \cdot \texttt{seq\_pair\_to\_leaves}[(s_1, s_2)]$$

The bilinearity (linear in each player's strategy holding the other fixed) is the key property that makes sequence-form strategies amenable to linear programming and best-response computation. The seq_pair_to_leaves dictionary represents the sequence form payoff matrix ($A$ in our lectures). Since this matrix is often sparse, we don't contruct it explicitly; instead, we only store the non-zero entries by this lookup table.

## Section 2 — Behavioral vs Sequence-Form Strategies

A strategy for a player is a vector of length `num_sequences`. Index 0 corressponds to the empty sequence. Often times, we will have to convert between sequence form and behavioral form strategies. In order to save space, we will also represent behavioral strategies by a vector of length `num_sequences` and index action probabilities the same way that we do sequences. The empty sequence isn't really required, so we allow it to be any number (by convention you could set it to 1 or 0, here, we chose 0).

### Behavioral strategy

At each information set $i$, the player chooses a probability distribution over their $k_i$ actions. If $b$ is a vector containing a behavioral strategy, and $i$ is the index of an infoset, then the slice `b[start_seq[i] : end_seq[i]]` is a distribution and sums to 1. 

$$\sum_{s=\texttt{start}[i]}^{\texttt{end}[i]-1} b[s] = 1 \quad \forall i$$

This is the natural representation for an agent: "given I'm at this information set, what distribution over actions do I play?"

### Sequence-form strategy

Entry $x[s]$ is the **unconditional probability** that the player plays the full path of actions leading to sequence $s$ starting from the root, in isolation from the contributions from chance and the opponent. Index 0 is fixed to always be 1. For each info set:

$$x[0] = 1, \qquad \sum_{s=\texttt{start}[i]}^{\texttt{end}[i]-1} x[s] = x[\texttt{parent}[i]] \quad \forall i$$

This is the right representation for payoff computation and optimisation because of bilinearity --- recall that $\langle x, Ay\rangle$ gives us the expected payoff if $x, y$ are sequence form strategies and $A$ is the sequence form payoff matrix.

### Conversion

- **Behavioral → sequence form**: multiply each action's probability by its parent's sequence-form value, top-down.
  $$x[s] = b[s] \cdot x[\texttt{parent\_seq}[i(s)]]$$
- **Sequence form → behavioral**: divide by parent's sequence-form value, bottom-up (with uniform fallback when parent = 0).

`TreeplexGame` provides `agent_to_behavioral_strategy` to convert any `Agent` to a behavioral vector. You are encouraged to read how these two functions are implemented to strenghten your understanding of the treeplex.

## Section 3 — The Solver: Payoffs and Reward Vector

The `Solver` class wraps a `TreeplexGame` and provides the key computational primitives. Two methods are already implemented.

### `compute_payoffs_from_seq_form_strategies(x1, x2)`

Evaluates the bilinear form:
$$u_1(x_1, x_2) = \sum_{(s_1, s_2)} x_1[s_1] \cdot x_2[s_2] \cdot \texttt{payoff}(s_1, s_2)$$

Both inputs must be in **sequence form**.

### `compute_reward_vector(player_id, opponent_seq_form_strategy)`

Fixes the opponent's strategy and returns a vector $r$ over the player's own sequences:
$$r[s_1] = \sum_{s_2} x_2[s_2] \cdot \texttt{payoff}(s_1, s_2) \quad \text{(for Player 1)}$$

Then $u_1(x_1, x_2) = r^\top x_1$ for any $x_1$. Player 2's reward vector negates the sign (Player 2 minimises).

Note that $r$ is computed over **all** sequences. Sequences that never appear in `seq_pair_to_leaves` simply receive zero reward.

## Section 4 — Task: `compute_best_response`

Given a fixed opponent sequence-form strategy, the best response for the player is the strategy that maximises their expected payoff. Thanks to the treeplex structure, this can be computed in $O(n)$ — no LP required.

### Algorithm (backward induction on the treeplex)

Start with the reward vector `r` from `compute_reward_vector`. Then process info sets in the treeplex in **reverse order** (why?):

For each info set $i$ (processing from last to first):
1. Find the best action: `best = argmax r[start[i] : end[i]]`
2. Propagate its value to the parent: `r[parent[i]] += r[best]`
3. Record the deterministic choice: set `vec[best] = 1.0`, all others in `[start, end)` to `0.0`

After the loop, `vec` holds a **behavioral** best-response strategy (deterministic). Optionally convert to sequence form with `convert_behavioral_to_sequence_form_strategy`.

**Why reversed order?** Info sets are discovered in DFS order (top-down). Reversed order gives bottom-up processing: by the time we handle info set $i$, all its descendant info sets have already been processed and their values propagated upward.

### Your task

Open `student_attempts/solver.py` and implement `compute_best_response`. The reward vector computation is already done for you at the top of the method. You are encouraged to use it as a reference if you are stuck.

```python
    def compute_best_response(
            self,
            player_id: int,
            opponent_seq_form_strategy: np.ndarray,
            convert_to_seq_form: bool = False) -> np.ndarray:
        
        """
        Args:
            player_id: PLAYER_1 or PLAYER_2, indicating which player is the best-responding one
            opponent_seq_form_strategy: A numpy array representing the opponent's strategy in sequence form.
            convert_to_seq_form: If True, convert the resulting best response from behavioral form to sequence form before returning. 
            If False, return the best response in behavioral form. By default, return behavioral strategies.
        
        Returns:
            numpy array representing the best response strategy for the given player against the opponent's strategy. 
            The returned strategy will be in behavioral form if convert_to_seq_form=False, and in sequence form if convert_to_seq_form=True.
        """

        vec = self.compute_reward_vector(player_id, opponent_seq_form_strategy)
        
        # Your code here: compute the best response strategy for player_id against the opponent's strategy given the reward vector.

        raise NotImplementedError("compute_best_response not implemented yet")
```

## Section 5 — Task: `saddle_point_residual`

In two-player zero-sum game, the **saddle-point residual** (also called *exploitability* of a strategy profile ($x_1, x_2$)) measures how far $(x_1, x_2)$ is from a Nash equilibrium.

Given $(x_1, x_2)$ in sequence form, define:
- $\text{BR}_1$ = best response of Player 1 against $x_2$
- $\text{BR}_2$ = best response of Player 2 against $x_1$
- $\text{upper} = u_1(\text{BR}_1,\; x_2)$ — the most Player 1 can gain by deviating
- $\text{lower} = u_1(x_1,\; \text{BR}_2)$ — Player 1's payoff when Player 2 plays their best response

$$\text{SPR}(x_1, x_2) = \text{upper} - \text{lower} \geq 0$$

At a Nash equilibrium neither player benefits from deviating, so `upper = lower = game value` and $\text{SPR} = 0$. For an approximate NE the SPR is small but always non-negative.

Note: `lower ≤ game value ≤ upper` always holds (minimax theorem), which guarantees SPR ≥ 0.

### Your task

Open `student_attempts/solver.py` and implement `saddle_point_residual`. 

Hint: It should call `compute_best_response` for each player and then call `compute_payoffs_from_seq_form_strategies` twice.

```python
    def saddle_point_residual(self, sol_p1: np.ndarray, sol_p2: np.ndarray):
        """
        Computes the saddle point residual (exploitability) of the given strategy profile. Note that the input strategies are expected to be in sequence form.

        Args:
            sol_p1: A numpy array representing player 1's strategy in sequence form.
            sol_p2: A numpy array representing player 2's strategy in sequence form.
        
        Returns:
            A float/numpy scalar representing the saddle point residual (exploitability) of the strategy profile (sol_p1, sol_p2).
        """

        # Your code here

        raise NotImplementedError("saddle_point_residual not implemented yet")
```

## Tips

- This should be a relatively simple task. Make sure you know how to navigate and iterate over sequences/treeplexes.
- If you encounter difficulties here, you are likely to face difficulties in the next Task. These functions may be used in the next task. If you are truly stuck, please seek help from the lecturer.